# Step 1 — inspect chunks for unusable material

This notebook covers only the first preparation step. We are not generating questions yet and we are not using embeddings or an LLM.

Goal: identify clearly unsuitable chunks: empty text, obviously unusable extraction, and reference-list material. Short passages, cover pages, tables, and captions are not automatically removed. Uncertain cases receive `review`.

Statuses: `exclude`, `review`, `eligible`. The original chunk remains under the `chunk` key. All automatic and manual screening information is stored under one `screening` key, separate from the source content.

## 1. Imports and input path

`Path` handles paths, `json` reads JSONL, `re` finds text patterns, `Counter` summarizes results, and `random` creates the later sample. `SAMPLE_INTERVAL` and `RANDOM_SEED` are defined once here and used in Step 2 so the sample is reproducible.

In [135]:
from pathlib import Path
import json
import re
from collections import Counter
import random

CHUNKS_PATH = Path('../data/processed/chunks.jsonl')
SAMPLE_INTERVAL = 6
RANDOM_SEED = 56


## 2. Load and inspect the JSONL schema

Each line is one JSON object. In the current ingestion output, the text is stored under `page_content` and provenance is nested under `metadata`. The earlier prototype incorrectly assumed a top-level `text` field; we correct that here. Always inspect the schema before writing checks.

In [136]:
def load_jsonl(path):
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

chunks = load_jsonl(CHUNKS_PATH)
print('number of chunks:', len(chunks))
print('top-level fields:', chunks[0].keys())
print('metadata fields:', chunks[0]['metadata'].keys())


number of chunks: 162
top-level fields: dict_keys(['metadata', 'page_content'])
metadata fields: dict_keys(['chunk_id', 'chunk_index', 'chunk_text_hash', 'creationdate', 'creator', 'document_hash', 'document_id', 'filename', 'page', 'page_label', 'producer', 'source', 'total_pages'])


## 3. Access text and provenance

These helpers hide the JSON structure from later code. `text_for` gets the extracted passage. `metadata_for` gets filename and page information. The first chunk is printed so we can see what the data actually contains.

In [137]:
def text_for(chunk):
    return str(chunk.get('page_content', '')).strip()

def metadata_for(chunk):
    return chunk.get('metadata', {})

first_chunk = chunks[0]
print(metadata_for(first_chunk).get('filename'))
print('page:', metadata_for(first_chunk).get('page_label'))
print(text_for(first_chunk)[:1000])


2024156785.pdf
page: 1
Predicting Football Match Outcomes Using Event Data and Machine Learning
Algorithms
Hassard, P., & Kerr, D. (2024). Predicting Football Match Outcomes Using Event Data and Machine Learning
Algorithms. In H. Zheng, I. Cleland, A. Moore, H. Wang, D. Glass, J. Rafferty, R. Bond, & J. Wallace (Eds.),
Proceedings of the 35th Irish Systems and Signals Conference, ISSC 2024 (Proceedings of the 35th Irish
Systems and Signals Conference, ISSC 2024). Institute of Electrical and Electronics Engineers Inc..
https://doi.org/10.1109/ISSC61953.2024.10603147
Link to publication record in Ulster University Research Portal
Published in:
Proceedings of the 35th Irish Systems and Signals Conference, ISSC 2024
Publication Status:
Published (in print/issue): 14/06/2024
DOI:
10.1109/ISSC61953.2024.10603147
Document Version
Publisher's PDF, also known as Version of record
For Author Accepted Manuscripts (AAM) published under Ulster University's Rights Retention Policy for Scholarly Work

## 4. Reference-list signals

A regex is just a text pattern. We use several weak signals: a heading such as `References`, repeated numeric citations such as `[12]`, repeated DOI/URL patterns, and many years.

A DOI or URL alone is not enough to exclude a chunk: normal article text and cover pages can contain them. The score is a signal, not proof. We inspect examples before trusting a threshold.

In [138]:
REFERENCE_HEADING = re.compile(r'\b(references|bibliography|literature cited)\b', re.IGNORECASE)
DOI_OR_URL = re.compile(r'(doi\s*:\s*|https?://|www\.)', re.IGNORECASE)
BRACKET_CITATION = re.compile(r'\[\d+(?:[,-]\d+)*\]')
YEAR = re.compile(r'\b(?:19|20)\d{2}\b')

def reference_score(text):
    """
    This provides a score for an input text. A higher score means a higher probability it is not usefull as a chunk
    """
    score = 0
    if REFERENCE_HEADING.search(text): score += 3
    if len(BRACKET_CITATION.findall(text)) >= 3: score += 1
    if len(DOI_OR_URL.findall(text)) >= 2: score += 1
    if len(YEAR.findall(text)) >= 4: score += 1
    return score


In [139]:
text = """
References

[1] Smith et al., 2020. doi:10.1234/abc
[2] Jones et al., 2021. https://example.com
[3] Brown et al., 2022.

this is a References: are blablabla in 1930 1931 1990 2001"""

print(REFERENCE_HEADING.search(text)) # This finds the word reference -> 3 points
print(BRACKET_CITATION.findall(text)) # This finds three citations -> +1 point
print(DOI_OR_URL.findall(text))       # This finds 2 url and dio -> +1 point
print(YEAR.findall(text))             # This finds 7 years -> +1 point
reference_score(text)

<re.Match object; span=(1, 11), match='References'>
['[1]', '[2]', '[3]']
['doi:', 'https://']
['2020', '2021', '2022', '1930', '1931', '1990', '2001']


6

### Reference-score examples

Expected behaviour:

- `References` followed by bibliography entries → strong signal;
- one data-source URL → weak/no signal;
- normal content with one citation and a year → should not be excluded.

The score is only a first heuristic. Inspect real chunks before choosing an exclusion threshold.

In [140]:
examples = {
    'reference_like': 'References\n[1] Smith (2020). Example.\n[2] Jones (2021). Another example.',
    'data_source': 'Our data came from https://example.org/data.',
    'content_with_citation': 'The model used features from 2019 [3].',
}

for name, example in examples.items():
    print(name, '->', reference_score(example))


reference_like -> 3
data_source -> 0
content_with_citation -> 0


## 5. Conservative chunk classification

This function performs only step 1. Empty text is excluded. Text with very few alphabetic characters is flagged for review, not automatically excluded, because tables and formulas may look unusual. A high reference score is excluded; a medium score is reviewed. No topic patterns are used here.

Each screening record has two top-level keys:

- `chunk`: the original chunk, including `page_content` and `metadata`;
- `screening`: automatic screening information, initially `status` and `flags`.

Keeping these separate means that we do not change the original chunk while investigating it. Later, `knowledge_base_decision`, `sampling_decision`, and `review_note` are added inside the same `screening` dictionary.

In [141]:
def classify_chunk(chunk):
    text = text_for(chunk)
    flags = []

    if not text:
        print("empty chunk found")
        return 'exclude', ['empty']

    letters = sum(character.isalpha() for character in text)
    if len(text) >= 40 and letters / len(text) < 0.20:
        flags.append('possibly_unusable_extraction')

    score = reference_score(text)
    if score >= 3:
        return 'exclude', ['reference_material']
    if score >= 2:
        flags.append('possibly_reference_material')

    return ('review' if flags else 'eligible'), flags

classified = []
for chunk in chunks:
    status, flags = classify_chunk(chunk)
    # create a list of dicts to investigate
    classified.append({
        'chunk': chunk,
        'screening': {'status': status, 'flags': flags},
    })

Counter(item['screening']['status'] for item in classified)


Counter({'eligible': 151, 'exclude': 6, 'review': 5})

## 6. Inspect automatic exclusions

Before sampling, inspect what the rules would exclude. If a real article section is removed, the heuristic is too aggressive. This is also where you learn whether reference pages in this corpus have a recognizable structure.

The automatic `status` and `flags` are signals, not final human decisions. They remain inside `screening` so we can compare them with the later manual decisions.

In [142]:
for item in classified:
    if item['screening']['status'] == 'exclude':
        metadata = metadata_for(item['chunk'])
        print('---')
        print(metadata.get('filename'), 'page', metadata.get('page_label'))
        print(item['screening']['flags'])
        print(text_for(item['chunk'])[:800])


---
2024156785.pdf page 7
['reference_material']
REFERENCES
[1] StatsBomb, “The 2015/16 Big 5 Leagues Free Data
Release.” Dec. 2023. https://statsbomb.com
[2] L. Breiman, “Random Forests,” Mach Learn, vol. 45,
no. 1, pp. 5–32, 2001.
[3] XGBoost Documentation. Accessed: Dec. 01, 2024.
https://xgboost.readthedocs.io
[4] C. A. Casal, et al, “Possession zone as a performance
indicator in football. The game of the best teams,”
Front Psychol , vol. 8, no. JUL, Jul. 2017, doi:
10.3389/fpsyg.2017.01176.
[5] C. A. Casal, et al “Possession in football: More than
a quantitative aspect - A mixed method study,” Front
Psychol, vol. 10, 2019: 10.3389/fpsyg.2019.00501.
[6] M. Klemp, et al, “In -play forecasting in football
using event and positional data,” Sci Rep, vol. 11, no.
1, Dec. 2021, doi: 10.1038/s41598-021-03157-3.
[7] Q. Zou, K. Song, and J. S
---
2024156785.pdf page 7
['reference_material']
Threats (xT)”, Accessed: Dec. 01, 2023. Available:
https://towardsai.net/p/sports-analytics-101-
expe

In [143]:
for item in classified:
    if item['screening']['status'] == 'review':
        metadata = metadata_for(item['chunk'])
        print('---')
        print(metadata.get('filename'), 'page', metadata.get('page_label'))
        print(item['screening']['flags'])
        print("chunk_id", metadata.get("chunk_id"))
        print(text_for(item['chunk'])[:800])


---
2024156785.pdf page 1
['possibly_reference_material']
chunk_id a42ab7c4d89ca0a595ab4488c354966a0a1f6da7393016f906d716cdcbc5b115
Predicting Football Match Outcomes Using Event Data and Machine Learning
Algorithms
Hassard, P., & Kerr, D. (2024). Predicting Football Match Outcomes Using Event Data and Machine Learning
Algorithms. In H. Zheng, I. Cleland, A. Moore, H. Wang, D. Glass, J. Rafferty, R. Bond, & J. Wallace (Eds.),
Proceedings of the 35th Irish Systems and Signals Conference, ISSC 2024 (Proceedings of the 35th Irish
Systems and Signals Conference, ISSC 2024). Institute of Electrical and Electronics Engineers Inc..
https://doi.org/10.1109/ISSC61953.2024.10603147
Link to publication record in Ulster University Research Portal
Published in:
Proceedings of the 35th Irish Systems and Signals Conference, ISSC 2024
Publication Status:
Published (in print/issue): 14/06/2024
DOI:
10.1109/ISSC61953.2024.10603147
Docum
---
a263-Article Text-847-2-10-20231219.pdf page 2
['possibly_refer

In [144]:
manual_reviewed = [
    {"chunk_id": "a42ab7c4d89ca0a595ab4488c354966a0a1f6da7393016f906d716cdcbc5b115", 
     "knowledge_base_decision": "include",
     "review_note": "Publisher metadata/cover; not suitable for synthetic-question sampling.",
     "sampling_decision": "exclude"},
    {"chunk_id": "efd11944e0d672a3adb0fba8898a1cc595717989e6b1e6918e400501d5c9a2b9", 
     "knowledge_base_decision": "include",
     "review_note": "Relevant content about model performance.",
     "sampling_decision": "include" },
    {"chunk_id": "e3a40d30948070291c2a52fb51ad2f11fb9b6684c3e4491ccb04e729207d5f59", 
     "knowledge_base_decision": "exclude",
     "review_note": "Confirmed reference list",
     "sampling_decision": "exclude"},
        {"chunk_id": "8fa79460636fa8fb27ed8cf2cc9cdc88df8b3a33e5a303f9c8e2a1c1b6df75f7", 
     "knowledge_base_decision": "include",
     "review_note": "Contains data of table, should not be included in sampling",
        "sampling_decision": "exclude"},
        {"chunk_id": "13c8384a7c4c15384e3c1c0ce8526dd6f489a05455ea33f83ad0e28b161eff02", 
     "knowledge_base_decision": "include",
     "review_note": "Table data without sufficient context for a synthetic question.",
        "sampling_decision": "exclude"}]                             

## 7. Saving screening decisions

We do not delete chunks from `chunks.jsonl`. Each record keeps the original data under `chunk` and stores all screening decisions under `screening`. This gives us an auditable derived representation without changing the ingestion output.

The final decisions answer two different questions:

- `knowledge_base_decision`: should this chunk be available to the future knowledge base?
- `sampling_decision`: should this chunk be used to generate synthetic questions?

A chunk can therefore be included in the knowledge base but excluded from synthetic-question sampling, such as metadata or a contextless table.

#### Step 1 — automatically include eligible chunks

In [145]:
selected_chunks = []
for item in classified:
    if item['screening']['status'] == "eligible":
        item['screening'].update({
            'knowledge_base_decision': 'include',
            'review_note': 'Automatically included; no exclusion or review flags.',
            'sampling_decision': 'include',
        })
        selected_chunks.append(item)

In [146]:
selected_chunks[0].keys()

dict_keys(['chunk', 'screening'])

In [147]:
selected_chunks[0]

{'chunk': {'metadata': {'chunk_id': '0b3c2953b48a5dcf224b30b79e6d5060faccc6afdbd80db0b583c78c5f6f8337',
   'chunk_index': 1,
   'chunk_text_hash': '257e2560feccf8ff2b071c82f69e0b3ce89e2ba2f7b873e54d3c129e5f867755',
   'creationdate': '2026-09-10T16:17:00+01:00',
   'creator': 'PyPDF',
   'document_hash': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
   'document_id': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
   'filename': '2024156785.pdf',
   'page': 0,
   'page_label': '1',
   'producer': 'OpenPDF 1.4.2',
   'source': 'data/raw/2024156785.pdf',
   'total_pages': 7},
  'page_content': "https://creativecommons.org/share-your-work/cclicenses/.\nTake down policy\nThe Research Portal is Ulster University's institutional repository that provides access to Ulster's research outputs. Every effort has been\nmade to ensure that content in the Research Portal does not infringe any person's rights, or applicable UK laws. If you discover content in\

#### Step 2 - Label items excluded from knowledge base

In [148]:
exclude_chunks = []
for item in classified:
    if item['screening']['status'] == "exclude":
        item['screening'].update({
            'knowledge_base_decision': 'exclude',
            'review_note': 'Automatically excluded; High exclusion score.',
            'sampling_decision': 'exclude',
        })
        exclude_chunks.append(item)

In [149]:
len(exclude_chunks)

6

In [150]:
exclude_chunks[0]

{'chunk': {'metadata': {'chunk_id': 'd4b510455a2df956a9a63c2b1b0891e12b0a97ebc5488fc4030aa508bc780628',
   'chunk_index': 24,
   'chunk_text_hash': 'd97bf52c1ac49daae5bd066adebeb3dea6ceed925c33e7450f339c60a9b00fa2',
   'creationdate': '2026-09-10T16:17:00+01:00',
   'creator': 'PyPDF',
   'document_hash': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
   'document_id': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
   'filename': '2024156785.pdf',
   'page': 6,
   'page_label': '7',
   'producer': 'OpenPDF 1.4.2',
   'source': 'data/raw/2024156785.pdf',
   'total_pages': 7},
  'page_content': 'REFERENCES\n[1] StatsBomb, “The 2015/16 Big 5 Leagues Free Data\nRelease.” Dec. 2023. https://statsbomb.com\n[2] L. Breiman, “Random Forests,” Mach Learn, vol. 45,\nno. 1, pp. 5–32, 2001.\n[3] XGBoost Documentation. Accessed: Dec. 01, 2024.\nhttps://xgboost.readthedocs.io\n[4] C. A. Casal, et al, “Possession zone as a performance\nindicator in football. T

#### Step 3 — Add manual decisions for review items

The manual review list is keyed by `chunk_id`. We find each original record and update only its `screening` dictionary. The `chunk` content is not changed.

In [151]:
reviewed_chunks = []
classified_by_id = {
    item['chunk']['metadata']['chunk_id']: item
    for item in classified
}

for review in manual_reviewed:
    item = classified_by_id[review['chunk_id']]
    item['screening'].update({
        'knowledge_base_decision': review['knowledge_base_decision'],
        'review_note': review['review_note'],
        'sampling_decision': review['sampling_decision'],
    })
    reviewed_chunks.append(item)

In [152]:
reviewed_chunks

[{'chunk': {'metadata': {'chunk_id': 'a42ab7c4d89ca0a595ab4488c354966a0a1f6da7393016f906d716cdcbc5b115',
    'chunk_index': 0,
    'chunk_text_hash': '4b5ff232a78c7191daf599977d6135e5d393c588844b6999eaa4e5709b707d57',
    'creationdate': '2026-09-10T16:17:00+01:00',
    'creator': 'PyPDF',
    'document_hash': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
    'document_id': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
    'filename': '2024156785.pdf',
    'page': 0,
    'page_label': '1',
    'producer': 'OpenPDF 1.4.2',
    'source': 'data/raw/2024156785.pdf',
    'total_pages': 7},
   'page_content': "Predicting Football Match Outcomes Using Event Data and Machine Learning\nAlgorithms\nHassard, P., & Kerr, D. (2024). Predicting Football Match Outcomes Using Event Data and Machine Learning\nAlgorithms. In H. Zheng, I. Cleland, A. Moore, H. Wang, D. Glass, J. Rafferty, R. Bond, & J. Wallace (Eds.),\nProceedings of the 35th Irish Systems and 

Let's make sure we the total of chunks still matches

In [153]:
screened_chunks = selected_chunks + reviewed_chunks + exclude_chunks
len(screened_chunks)

162

## 8. Save the screened chunks

In [154]:
import json
from pathlib import Path

In [155]:
output_path = Path("../data/processed/screened_chunks.jsonl")

with output_path.open("w", encoding="utf-8") as file:
    for record in screened_chunks:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

# Step 2

Now that the chunks are screened and saved, we create a random sample of passages from which synthetic questions can be generated.

There are 152 chunks with `sampling_decision: include`. Sampling one in six produces 25 passages because integer division rounds down: `152 // 6 == 25`.

Below we:
1. load all screened chunks.
2. select only chunks that are `include` in for `sampling_decision`
3. calculate the sample size with `SAMPLE_INTERVAL`;
4. use `RANDOM_SEED` with Python's `random` module to make the selection reproducible;
5. verify uniqueness and inspect the distribution across source documents.

In [156]:
input_path = Path("../data/processed/screened_chunks.jsonl")

# load the screened chunks
with input_path.open("r", encoding="utf-8") as file:
    screened_chunks = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

# select eligible chunks
eligible_chunks = [
    record for record in screened_chunks
    if record["screening"]["sampling_decision"] == "include"
]
print(f"There are {len(eligible_chunks)} chunks eligible for sampling")

# create a random sample from the list of eligible chunks
sample_size = len(eligible_chunks) // SAMPLE_INTERVAL
rng = random.Random(RANDOM_SEED)
sampled_chunks = rng.sample(eligible_chunks, k=sample_size)

sampled_ids = [record["chunk"]["metadata"]["chunk_id"] for record in sampled_chunks]
assert len(sampled_ids) == len(set(sampled_ids))

document_counts = Counter(
    record["chunk"]["metadata"]["filename"]
    for record in sampled_chunks
)

print(f"The random sample contains {len(sampled_chunks)} unique chunks")
print("Documents:", document_counts)
print("-" * 60)
print(sampled_chunks[0]["chunk"]["page_content"][:1000])

There are 152 chunks eligible for sampling
The random sample contains 25 unique chunks
Documents: Counter({'s10994-024-06625-9.pdf': 11, '2024156785.pdf': 5, 's10994-018-5704-6.pdf': 5, 'a263-Article Text-847-2-10-20231219.pdf': 4})
------------------------------------------------------------
complex models are indeed not always better (Hand, 2006; Gosiewska et al., 2021).
The bookmakers’ model came out top in the result prediction task (Task 2). This was
not unexpected, as bookmakers have access to comprehensive match data sets and are able
to tap into the wisdom of very large crowds of people—some with profound understanding
of soccer—who bet on the outcomes of matches through the betting platforms. Given the
minimal information about matches that we used in our study, one could have expected
that the margin between the bookmakers’ model and the machine learning models would
be even much higher, but it wasn’t—and this is rather motivating for machine learning
researchers!
Considering

In [157]:
sampled_output_path = Path("../data/processed/sampled_chunks.jsonl")

with sampled_output_path.open("w", encoding="utf-8") as file:
    for record in sampled_chunks:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(sampled_chunks)} sampled chunks to {sampled_output_path}")


Saved 25 sampled chunks to ../data/processed/sampled_chunks.jsonl


In [ ]:
reloaded_sample = load_jsonl(sampled_output_path)
assert len(reloaded_sample) == len(sampled_chunks)
assert {record["chunk"]["metadata"]["chunk_id"] for record in reloaded_sample} == set(sampled_ids)
print("Sample file verified successfully")
